In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
shlokraval_ppe_dataset_yolov8_path = kagglehub.dataset_download('shlokraval/ppe-dataset-yolov8')

print('Data source import complete.')


In [ ]:
import os
import yaml
from collections import Counter
import matplotlib.pyplot as plt
import random
import cv2

In [ ]:
import shutil
import os

# مكان الداتا الأصلية (read only)
path = "/kaggle/input/datasets/shlokraval/ppe-dataset-yolov8"

# المكان اللي هنشتغل عليه
new_path = "/kaggle/working/ppe_dataset"

# لو فيه نسخة قديمة امسحيها
if os.path.exists(new_path):
    shutil.rmtree(new_path)

# نسخ الداتا
shutil.copytree(path, new_path)

# من هنا اشتغلي على النسخة القابلة للتعديل
path = new_path

print("Working Path:", path)
print(os.listdir(path))

In [ ]:
yaml_path=os.path.join(path,'data.yaml')
with open(yaml_path,'r') as f:
    data=yaml.safe_load(f)
print(f"Number of Classes: {data['nc']}\n")

for i, class_name in enumerate(data["names"]):
    print(f"{i} --> {class_name}")


In [ ]:
splits = ["train", "valid", "test"]

missing_labels = 0
missing_images = 0
empty_labels = 0
invalid_classes = 0
invalid_boxes = 0

valid_extensions = [".jpg", ".jpeg", ".png"]

for split in splits:

    image_dir = os.path.join(path, split, "images")
    label_dir = os.path.join(path, split, "labels")

    image_files = {
        os.path.splitext(f)[0]
        for f in os.listdir(image_dir)
        if os.path.splitext(f)[1].lower() in valid_extensions
    }

    label_files = {
        os.path.splitext(f)[0]
        for f in os.listdir(label_dir)
        if f.endswith(".txt")
    }

    # -----------------------------
    # Missing Labels
    # -----------------------------
    for image in image_files:

        if image not in label_files:
            missing_labels += 1

    # -----------------------------
    # Missing Images
    # -----------------------------
    for label in label_files:

        if label not in image_files:
            missing_images += 1

    # -----------------------------
    # Check Labels
    # -----------------------------
    for label in label_files:

        label_path = os.path.join(label_dir, label + ".txt")

        # Empty Label
        if os.path.getsize(label_path) == 0:
            empty_labels += 1
            continue

        with open(label_path, "r") as f:

            for line_num, line in enumerate(f, start=1):

                values = line.strip().split()

                if len(values) != 5:
                    continue

                class_id = int(values[0])

                # Invalid Class ID
                if class_id < 0 or class_id >= data["nc"]:
                    invalid_classes += 1

                x, y, w, h = map(float, values[1:])

                # Invalid Bounding Box
                if not (
                    0 <= x <= 1 and
                    0 <= y <= 1 and
                    0 <= w <= 1 and
                    0 <= h <= 1
                ):


                    invalid_boxes += 1


print("\n" + "=" * 45)
print("          DATASET VALIDATION REPORT")
print("=" * 45)

print(f"Missing Labels      : {missing_labels}")
print(f"Missing Images      : {missing_images}")
print(f"Empty Labels        : {empty_labels}")
print(f"Invalid Class IDs   : {invalid_classes}")
print(f"Invalid BoundingBox : {invalid_boxes}")

print("=" * 45)

if (missing_labels == 0 and
    missing_images == 0 and
    empty_labels == 0 and
    invalid_classes == 0 and
    invalid_boxes == 0):

    print("✅ Dataset Validation PASSED")

else:

    print("⚠️ Dataset contains issues. Please review the messages above.")

In [ ]:
empty_files = []

for split in splits:

    label_dir = os.path.join(path, split, "labels")

    for file in os.listdir(label_dir):

        label_path = os.path.join(label_dir, file)

        if os.path.getsize(label_path) == 0:
            empty_files.append((split, file))

print("Number of Empty Labels:", len(empty_files))

# اعرض أول 5 صور
for split, file in random.sample(empty_files, 5):

    image_name = os.path.splitext(file)[0] + ".jpg"

    image_path = os.path.join(path, split, "images", image_name)

    if not os.path.exists(image_path):
        image_name = os.path.splitext(file)[0] + ".png"
        image_path = os.path.join(path, split, "images", image_name)

    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.title(f"{split} - {image_name}")
    plt.axis("off")
    plt.show()

In [ ]:
deleted_images = 0
deleted_labels = 0

valid_extensions = [".jpg", ".jpeg", ".png"]

for split in splits:

    image_dir = os.path.join(path, split, "images")
    label_dir = os.path.join(path, split, "labels")

    for label_file in os.listdir(label_dir):

        if not label_file.endswith(".txt"):
            continue

        label_path = os.path.join(label_dir, label_file)

        # لو الليبل فاضي
        if os.path.getsize(label_path) == 0:

            # حذف الليبل
            os.remove(label_path)
            deleted_labels += 1

            # حذف الصورة المقابلة
            image_name = os.path.splitext(label_file)[0]

            for ext in valid_extensions:

                image_path = os.path.join(image_dir, image_name + ext)

                if os.path.exists(image_path):
                    os.remove(image_path)
                    deleted_images += 1
                    break

print("=" * 40)
print("Deleted Empty Labels :", deleted_labels)
print("Deleted Images       :", deleted_images)
print("=" * 40)

In [ ]:
missing_labels = 0
missing_images = 0
empty_labels = 0
invalid_classes = 0
invalid_boxes = 0

valid_extensions = [".jpg", ".jpeg", ".png"]

for split in splits:

    image_dir = os.path.join(path, split, "images")
    label_dir = os.path.join(path, split, "labels")

    image_files = {
        os.path.splitext(f)[0]
        for f in os.listdir(image_dir)
        if os.path.splitext(f)[1].lower() in valid_extensions
    }

    label_files = {
        os.path.splitext(f)[0]
        for f in os.listdir(label_dir)
        if f.endswith(".txt")
    }

    # -----------------------------
    # Missing Labels
    # -----------------------------
    for image in image_files:

        if image not in label_files:
            missing_labels += 1

    # -----------------------------
    # Missing Images
    # -----------------------------
    for label in label_files:

        if label not in image_files:
            missing_images += 1

    # -----------------------------
    # Check Labels
    # -----------------------------
    for label in label_files:

        label_path = os.path.join(label_dir, label + ".txt")

        # Empty Label
        if os.path.getsize(label_path) == 0:
            empty_labels += 1
            continue

        with open(label_path, "r") as f:

            for line_num, line in enumerate(f, start=1):

                values = line.strip().split()

                if len(values) != 5:
                    continue

                class_id = int(values[0])

                # Invalid Class ID
                if class_id < 0 or class_id >= data["nc"]:
                    invalid_classes += 1

                x, y, w, h = map(float, values[1:])

                # Invalid Bounding Box
                if not (
                    0 <= x <= 1 and
                    0 <= y <= 1 and
                    0 <= w <= 1 and
                    0 <= h <= 1
                ):


                    invalid_boxes += 1


print("\n" + "=" * 45)
print("          DATASET VALIDATION REPORT")
print("=" * 45)

print(f"Missing Labels      : {missing_labels}")
print(f"Missing Images      : {missing_images}")
print(f"Empty Labels        : {empty_labels}")
print(f"Invalid Class IDs   : {invalid_classes}")
print(f"Invalid BoundingBox : {invalid_boxes}")

print("=" * 45)

if (missing_labels == 0 and
    missing_images == 0 and
    empty_labels == 0 and
    invalid_classes == 0 and
    invalid_boxes == 0):

    print("✅ Dataset Validation PASSED")

else:

    print("⚠️ Dataset contains issues. Please review the messages above.")

In [ ]:
print("Dataset Size")

for split in splits:
    image_path = os.path.join(path, split, "images")

    num_images = len([
        img for img in os.listdir(image_path)
        if img.endswith((".jpg", ".jpeg", ".png"))
    ])

    print(f"{split}: {num_images} images")

In [ ]:
label_counts = Counter()

for split in splits:

    label_dir = os.path.join(path, split, "labels")

    for file in os.listdir(label_dir):

        if file.endswith(".txt"):

            with open(os.path.join(label_dir, file), "r") as f:

                for line in f:
                    class_id = int(line.split()[0])
                    label_counts[class_id] += 1

print("Class Distribution:")

for class_id, count in sorted(label_counts.items()):
    print(f"{data['names'][class_id]} : {count}")

In [ ]:
classes = [data["names"][i] for i in sorted(label_counts.keys())]
counts = [label_counts[i] for i in sorted(label_counts.keys())]

plt.figure(figsize=(12,5))
plt.bar(classes, counts)

plt.xticks(rotation=45)
plt.xlabel("Classes")
plt.ylabel("Number of Objects")
plt.title("Class Distribution")

plt.show()

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

# أسماء الكلاسات
class_names = data["names"]

# مسارات الصور والليبلز
image_dir = os.path.join(path, "train", "images")
label_dir = os.path.join(path, "train", "labels")

# اختيار 10 صور عشوائية
image_files = random.sample(
    [f for f in os.listdir(image_dir) if f.endswith((".jpg", ".jpeg", ".png"))],
    10
)

for image_file in image_files:

    image_path = os.path.join(image_dir, image_file)
    label_path = os.path.join(label_dir, image_file.rsplit(".", 1)[0] + ".txt")

    # قراءة الصورة
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    # قراءة الليبل
    with open(label_path, "r") as f:
        labels = f.readlines()

    # رسم الـ Bounding Boxes واسم الكلاس
    for label in labels:

        class_id, x_center, y_center, bw, bh = map(float, label.split())

        class_id = int(class_id)

        # تحويل إحداثيات YOLO إلى Pixels
        x_center *= w
        y_center *= h
        bw *= w
        bh *= h

        x1 = int(x_center - bw / 2)
        y1 = int(y_center - bh / 2)
        x2 = int(x_center + bw / 2)
        y2 = int(y_center + bh / 2)

        # رسم الـ Bounding Box
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # كتابة اسم الكلاس
        cv2.putText(
            image,
            class_names[class_id],
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2
        )

    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.title(image_file)
    plt.show()

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
model = YOLO("yolo26n.pt")

In [ ]:
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=512,
    batch=32,

    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    patience=20,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2,

    fliplr=0.5,
    flipud=0.0,

    mosaic=1.0,
    mixup=0.1,

    device=0,
    workers=4,

    pretrained=True,

    project="Safety_Project",
    name="PPE_YOLO26",

    save=True,
    plots=True,
    verbose=True,
    save_period=1
)

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/Safety_Project/PPE_YOLO26/weights/last.pt")

model.train(
    resume=True
)

In [ ]:
import os
import shutil
from glob import glob

# مكان التدريب
run_dir = "/kaggle/working/Safety_Project/PPE_YOLO26"

# فولدر هنجمع فيه كل الـ weights
export_dir = "/kaggle/working/exported_weights"

os.makedirs(export_dir, exist_ok=True)

# البحث عن جميع ملفات .pt
weight_files = glob(os.path.join(run_dir, "weights", "*.pt"))

print("=" * 50)
print("Saved Weights")
print("=" * 50)

for file in weight_files:
    print(os.path.basename(file))
    shutil.copy(file, export_dir)

print("\nAll weights copied to:")
print(export_dir)

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/PPE_YOLO26_weights",
    "zip",
    "/kaggle/working/exported_weights"
)

print("ZIP File Created:")
print("/kaggle/working/PPE_YOLO26_weights.zip")

In [ ]:
import os

for folder in [
    "/kaggle/working/runs/detect/Safety_Project/PPE_YOLO26",
    "/kaggle/working/runs/detect/Safety_Project/PPE_YOLO26-2",
    "/kaggle/working/runs/detect/Safety_Project/PPE_YOLO26_Finetune",
]:
    print("=" * 70)
    print(folder)
    print("Exists:", os.path.exists(folder))

    weights = os.path.join(folder, "weights")
    print("Weights folder:", os.path.exists(weights))

    if os.path.exists(weights):
        print(os.listdir(weights))

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/Safety_Project/PPE_YOLO26/weights/best.pt")

metrics = model.val(data=yaml_path)

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/Safety_Project/PPE_YOLO26/weights/best.pt")

results = model.train(
    data=yaml_path,

    epochs=20,          # هيكمل 30 Epoch كمان
    imgsz=512,          # رجعيها 640
    batch=32,

    lr0=0.0001,
    lrf=0.01,

    optimizer="SGD",
    momentum=0.937,
    weight_decay=0.0005,

    patience=10,

    device=0,

    project="Safety_Project",
    name="PPE_YOLO26_Finetune",

    pretrained=False,
    save=True,
    plots=True
)

In [ ]:
import shutil

weights_folder = "/kaggle/working/runs/detect/Safety_Project/PPE_YOLO26/weights"

shutil.make_archive(
    "/kaggle/working/PPE_YOLO26_weights",
    "zip",
    weights_folder
)

print("Done!")
print("/kaggle/working/PPE_YOLO26_weights2.zip")

In [ ]:
import os

folder = "/kaggle/working/runs/detect/Safety_Project/PPE_YOLO26"

print("Exists:", os.path.exists(folder))